## Pore size and flow rate distributions in 2D porous media calculator

In [ ]:
import os
import numpy as np
import pyvista as pv
from scipy.ndimage import distance_transform_edt
from scipy.spatial import KDTree
from skimage.morphology import medial_axis
from shapely import vectorized
from shapely.geometry import Point, LineString, box
from shapely.ops import polygonize, unary_union
from shapely.prepared import prep
import matplotlib.pyplot as plt

In [ ]:
import os
import glob
import pyvista as pv

print("Loading OpenFOAM case and auto-detecting patches...")

# read case directory from bash environment, or use default for manual testing
case_dir = os.environ.get("CASE_DIR", os.path.expanduser("~/OpenFOAM/jose-v2406/run/YDRAY-flow_n13_sat"))

# automatically find the .vtm file inside the VTK folder
# (ignoring the .vtm.series file)
vtk_search_path = os.path.join(case_dir, "VTK", "*.vtm")
vtm_files = [f for f in glob.glob(vtk_search_path) if not f.endswith('.series')]

if not vtm_files:
    raise FileNotFoundError(f"No .vtm file found in {os.path.join(case_dir, 'VTK')}")

# take the last one (which will be the latest time step generated by bash)
vtk_file = sorted(vtm_files)[-1]

print(f"Loading case from: {case_dir}")
print(f"Detected VTK file: {vtk_file}")

mb = pv.read(vtk_file)
vol_mesh = mb["internal"]

# auto-detect the obstacle patch name
boundary_blocks = mb["boundary"]
common_wall_names = ["wallFluidSolid", "wall", "cylinders", "obstacles", "grainWalls"]

cyl_patch = None


for name in common_wall_names:
    if hasattr(boundary_blocks, 'keys') and name in boundary_blocks.keys():
        cyl_patch = boundary_blocks[name]
        print(f"Obstacle patch detected: '{name}'")
        break

if cyl_patch is None and hasattr(boundary_blocks, 'keys'):
    print("Standard patch not found. Searching for individual cylinder patches...")
    cyl_keys = [k for k in boundary_blocks.keys() if k.startswith('cyl')]
    
    if cyl_keys:
        print(f"Found {len(cyl_keys)} individual cylinder patches. Merging them into a single object...")
        cyl_patch = boundary_blocks[cyl_keys[0]].extract_surface()
        
        for key in cyl_keys[1:]:
            cyl_patch = cyl_patch.merge(boundary_blocks[key].extract_surface())
            
        print("Merge complete! Proceeding with unified obstacle patch.")

if cyl_patch is None:
    available_names = boundary_blocks.keys() if hasattr(boundary_blocks, 'keys') else "Unknown"
    raise KeyError(
        f"Could not find expected patches {common_wall_names}, nor any 'cyl*' pattern.\n"
        f"AVAILABLE PATCHES IN THIS VTK ARE: {available_names}"
    )

# calculate Z bounds and extract 2D slice
_, _, _, _, zmin, zmax = vol_mesh.bounds
z_mid = 0.5 * (zmin + zmax)

obst_section = (
    cyl_patch.extract_surface()
             .slice(normal="z", origin=(0, 0, z_mid))
             .clean()
)
dom_section = (
    vol_mesh.extract_surface()
            .slice(normal="z", origin=(0, 0, z_mid))
            .clean()
)

xmin, xmax, ymin, ymax, _, _ = dom_section.bounds
print(f"Geometry slice extracted at Z = {z_mid:.4f}")

RAW_DATA_DIR = os.environ.get('RAW_DATA_DIR', './')
FIGURES_DIR = os.environ.get('FIGURES_DIR', './')

print(f"Saving data in: {RAW_DATA_DIR}")
print(f"Saving figures in: {FIGURES_DIR}")

In [ ]:
# Import the function to find peaks
from skimage.feature import peak_local_max

print("Starting geometry analysis...")

# Polygonize the obstacles:

# Extract the 2D points and lines from the obstacle section
points_2d = obst_section.points[:, :2]
lines_array = obst_section.lines

shapely_lines = []
i = 0
while i < len(lines_array):
    n_points = lines_array[i]
    point_indices = lines_array[i + 1 : i + 1 + n_points]
    line_coords = points_2d[point_indices]
    
    # Ensure the line has at least 2 points
    if len(line_coords) > 1:
        shapely_lines.append(LineString(line_coords))
    
    i += n_points + 1

# We use polygonize to find all closed polygons
all_polygons = list(polygonize(shapely_lines))
print(f"Found {len(all_polygons)} initial polygons.")

## Classify polygons (circular vs. merged)
# A perfect circle has a circularity of 1.
# Circularity = (4 * pi * Area) / (Perimeter^2)
CIRCULARITY_THRESHOLD = 0.9 

circular_polygons = []
merged_polygons = []

for poly in all_polygons:
    if poly.area == 0:
        continue
    
    circularity = (4 * np.pi * poly.area) / (poly.length ** 2)
    
    if circularity > CIRCULARITY_THRESHOLD:
        circular_polygons.append(poly)
    else:
        merged_polygons.append(poly)

print(f"Classified: {len(circular_polygons)} individual circles and {len(merged_polygons)} merged polygons.")

## Decompose merged polygons
# We use a distance transform.

# Define a grid to rasterize the polygons
GRID_RES = 500 # Resolution
x_grid = np.linspace(xmin, xmax, GRID_RES)
y_grid = np.linspace(ymin, ymax, GRID_RES)
xx, yy = np.meshgrid(x_grid, y_grid)

# Flattened grid points for checking
grid_points_x = xx.ravel()
grid_points_y = yy.ravel()

# Calculate the pixel size (average, assuming similar proportions)
pixel_size = (x_grid[1] - x_grid[0] + y_grid[1] - y_grid[0]) / 2.0

# Estimate a minimum distance between peaks
# If individual circles are found, we use their radius as a guide
if circular_polygons:
    radii = [np.sqrt(p.area / np.pi) for p in circular_polygons]
    median_radius = np.median(radii)
    # Minimum distance = 80% of the median radius, converted to pixels
    min_dist_px = int((median_radius * 0.8) / pixel_size)
    min_dist_px = max(1, min_dist_px) # Ensure it is at least 1
else:
    # If no circles are found, use a fixed value (adjust if necessary)
    min_dist_px = 5 
    print("Warning: No individual circles found. Using min_distance=5px for peak detection.")


# Final list to store all circles (original + decomposed)
final_circular_polygons = list(circular_polygons)

for i, merged_poly in enumerate(merged_polygons):
    print(f"Processing merged polygon {i+1}/{len(merged_polygons)}...")
    
    # Rasterize: create a 2D mask of the polygon
    mask_flat = vectorized.contains(merged_poly, grid_points_x, grid_points_y)
    mask = mask_flat.reshape(GRID_RES, GRID_RES)
    
    if not np.any(mask):
        print(f"  Polígono {i+1} está vacío o fuera de los límites, omitiendo.")
        continue

    # Distance transform: distance from each internal pixel to the boundary
    distance = distance_transform_edt(mask)
    
    # Find peaks (circle centers)
    # peak_local_max finds (row, column) coordinates of local maxima
    local_max_indices = peak_local_max(distance, min_distance=min_dist_px, labels=mask)
    
    # Recreate circles
    for (row, col) in local_max_indices:
        # Convert pixel indices (row, col) to coordinates (x, y)
        center_x = x_grid[col]
        center_y = y_grid[row]
        
        # The radius is the value of the distance transform at the peak
        radius_px = distance[row, col]
        radius = radius_px * pixel_size
        
        # Create the new circle and add it to the final list
        new_circle = Point(center_x, center_y).buffer(radius)
        final_circular_polygons.append(new_circle)

print(f"Decomposition complete. Total circles: {len(final_circular_polygons)}")

## Visualization
print("Generating visualization...")

fig, ax = plt.subplots(figsize=(12, 12), dpi = 300)

# Plot original merged polygons (for comparison)
for poly in merged_polygons:
    x, y = poly.exterior.xy
    ax.plot(x, y, color='red', linestyle='--', linewidth=1.5, label='Original' if 'Original' not in ax.get_legend_handles_labels()[1] else "")

# Plot the final list of circles
for poly in final_circular_polygons:
    x, y = poly.exterior.xy
    ax.fill(x, y, alpha=0.6, fc='blue', ec='none', label='Decomposed circles' if 'Decomposed circles' not in ax.get_legend_handles_labels()[1] else "")

ax.set_aspect('equal')
ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)
ax.set_title(f"Obstacle decomposition ({len(final_circular_polygons)} final circles)")
ax.legend()
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()

In [ ]:
print("Defining Region of Interest (ROI)...")

# Calculate total domain length in X
x_length = xmax - xmin

# Define new ROI boundaries (ignoring 3% at each end)
roi_xmin = xmin + 0.03 * x_length
roi_xmax = xmax - 0.03 * x_length

# Y boundaries remain unchanged
roi_ymin = ymin
roi_ymax = ymax

print(f"Original X bounds: ({xmin:.4f}, {xmax:.4f})")
print(f"ROI X bounds:      ({roi_xmin:.4f}, {roi_xmax:.4f})")

In [ ]:
import pandas as pd
from scipy.spatial import Delaunay
from scipy.spatial import cKDTree

print("Starting tube network construction...")

## Extract centers and radii

bead_data = []
for poly in final_circular_polygons:
    bead_data.append({
        'center': np.array([poly.centroid.x, poly.centroid.y]),
        'radius': np.sqrt(poly.area / np.pi)
    })
centers = np.array([d['center'] for d in bead_data])
radii = np.array([d['radius'] for d in bead_data])
n_real_beads = len(centers)
print(f"Extracted {n_real_beads} grain centers and radii.")

## Create ghost beads for Ymin / Ymax boundaries

max_r = np.max(radii)
reflection_dist = 5.0 * max_r
near_bottom_indices = np.where(centers[:, 1] < ymin + reflection_dist)[0]
ghost_centers_bottom = np.copy(centers[near_bottom_indices])
ghost_centers_bottom[:, 1] = ymin - (ghost_centers_bottom[:, 1] - ymin) # Reflejar
ghost_radii_bottom = radii[near_bottom_indices]
near_top_indices = np.where(centers[:, 1] > ymax - reflection_dist)[0]
ghost_centers_top = np.copy(centers[near_top_indices])
ghost_centers_top[:, 1] = ymax + (ymax - ghost_centers_top[:, 1]) # Reflejar
ghost_radii_top = radii[near_top_indices]
all_centers = np.vstack((centers, ghost_centers_bottom, ghost_centers_top))
all_radii = np.hstack((radii, ghost_radii_bottom, ghost_radii_top))
print(f"Created {len(ghost_centers_bottom)} bottom ghosts and {len(ghost_centers_top)} top ghosts.")

## Delaunay triangulation

tri = Delaunay(all_centers)
edges = set()
for simplex in tri.simplices:
    edges.add(tuple(sorted((simplex[0], simplex[1]))))
    edges.add(tuple(sorted((simplex[1], simplex[2]))))
    edges.add(tuple(sorted((simplex[2], simplex[0]))))
print(f"Delaunay triangulation complete. {len(edges)} total edges found.")

## Prepare KDTree for fast sampling

print("Preparing KDTree for fast velocity sampling...")
VEL_NAME_CELL = None
common_names = ['U', 'velocity', 'Velocity', 'VELOCITY']
for name in common_names:
    if name in vol_mesh.cell_data:
        VEL_NAME_CELL = name
        break
if VEL_NAME_CELL is None:
    print(f"WARNING: 'U' or 'velocity' not found in vol_mesh.cell_data.")
    print("Searching for the first available vector field in cell_data...")
    for name, array in vol_mesh.cell_data.items():
        if array.ndim == 2 and array.shape[1] == 3:
            VEL_NAME_CELL = name
            break
if VEL_NAME_CELL:
    print(f"Using cell velocity field: '{VEL_NAME_CELL}'")
    cell_centers_3d = vol_mesh.cell_centers().points
    U_cells = vol_mesh.cell_data[VEL_NAME_CELL][:, :2] # (M,2)
    tree = cKDTree(cell_centers_3d[:, :2])
    print("KDTree built.")
else:
    print("FATAL ERROR: No velocity field found in cell_data. Flow rates will be 0.")
    tree = None


## Calculate Tube Properties (ROI FILTERED & NO DUPLICATES)

H_METERS = 0.001 # 1 mm in meters
N_SAMPLES = 50   # Number of integration points
tube_list = []

if 'tree' not in locals() or tree is None:
    print("ERROR: KDTree is not defined. Please run the tree preparation cell.")
if 'roi_xmin' not in locals():
    print("ERROR: ROI boundaries are not defined. Please run the ROI definition cell.")

wall_tubes_created_for_bead = set()
print(f"Starting tube calculation (filtering by X-ROI: [{roi_xmin:.4f}, {roi_xmax:.4f}])...")

# Threshold to define "real flow"
# Used to calculate the effective width
VEL_THRESHOLD_RATIO = 0.001 # 5% of the profile's maximum velocity

for idx1, idx2 in edges:
    
    # Skip ghost-to-ghost edges
    if idx1 >= n_real_beads and idx2 >= n_real_beads:
        continue

    # Tube geometry
    is_wall_tube = (idx1 >= n_real_beads) or (idx2 >= n_real_beads)
    
    if is_wall_tube:
        real_idx = idx1 if idx1 < n_real_beads else idx2
        if real_idx in wall_tubes_created_for_bead:
            continue
        c_real = centers[real_idx]
        r_real = radii[real_idx]
        is_top_wall = c_real[1] > (ymax + ymin) / 2.0
        
        if is_top_wall:
            p_edge_bead = np.array([c_real[0], c_real[1] + r_real])
            p_edge_wall = np.array([c_real[0], ymax])
            n_vec = np.array([0.0, 1.0])
        else:
            p_edge_bead = np.array([c_real[0], c_real[1] - r_real])
            p_edge_wall = np.array([c_real[0], ymin])
            n_vec = np.array([0.0, -1.0])
            
        width = np.linalg.norm(p_edge_bead - p_edge_wall)
        wall_tubes_created_for_bead.add(real_idx)
        
    else:
        c1, c2 = all_centers[idx1], all_centers[idx2]
        r1, r2 = all_radii[idx1], all_radii[idx2]
        center_vec = c2 - c1
        dist = np.linalg.norm(center_vec)
        width = dist - (r1 + r2)
        n_vec = center_vec / dist
        p_edge_wall = c1 + n_vec * r1
        p_edge_bead = c2 - n_vec * r2

    # Filter overlapping tubes
    if width <= 0:
        continue

    # Calculate flow rate and effective width
    n_vec = np.array([-n_vec[1], n_vec[0]]) # normal vector for the integration
    flow_rate = 0.0
    effective_width = width # default to geometric width
    
    
    if tree is not None and width > 1e-9: 
        # Sampling points across the GEOMETRIC width
        sample_points_2d = np.linspace(p_edge_wall, p_edge_bead, N_SAMPLES)
        # Integration axis (distance from 0 to width)
        integration_axis = np.linspace(0, width, N_SAMPLES)
        
        # Sample velocities
        dists, idxs = tree.query(sample_points_2d, k=1)
        velocities_2d = U_cells[idxs]
        velocities_2d = np.nan_to_num(velocities_2d, nan=0.0, posinf=0.0, neginf=0.0)

        

        # Velocity profile normal to the tube
        v_normal_components = np.dot(velocities_2d, n_vec)
        v_abs = np.abs(v_normal_components)
        v_max = np.max(v_abs)
        
        if v_max > 1e-16: # if there is any flow
            threshold = v_max * VEL_THRESHOLD_RATIO
            
            # Find the indices (within the N_SAMPLES array) where the flow is "real"
            effective_indices = np.where(v_abs > threshold)[0]
            
            if len(effective_indices) > 1:
                # Find the start and end of the flow
                i_start = np.min(effective_indices)
                i_end = np.max(effective_indices)
                
                # Recalculate flow rate (integral) ONLY within the effective section
                v_effective = v_normal_components[i_start:i_end+1]
                axis_effective = integration_axis[i_start:i_end+1]
                
                integral_value = np.trapz(y=v_effective, x=axis_effective)
                flow_rate = np.abs(integral_value) * H_METERS
                
                    
                # KEY: correct the tube width
                effective_width = axis_effective[-1] - axis_effective[0]
                
            else:
                # entire flow is below threshold, or only a single point was found
                flow_rate = 0.0
                effective_width = 0.0
                
        else:
            # No flow detected
            flow_rate = 0.0
            effective_width = 0.0 # blocked tube
            
   

    # compute midpoint
    midpoint_2d = (p_edge_wall + p_edge_bead) / 2.0
    
    # save results
    tube_list.append({
        'midpoint': midpoint_2d, 
        'width': effective_width, 
        'flow_rate': flow_rate,
        'is_wall_tube': is_wall_tube, 
        'bead_indices': (idx1, idx2)
    })

print(f"\n--- Process completed ---")
print(f"Found {len(tube_list)} valid tubes (width > 0, no duplicates, and within the ROI).")

df_tubos = pd.DataFrame(tube_list)
print("\nFirst 5 tubes found:")
print(df_tubos.head())

In [ ]:
print("Generating distribution histograms (with range limited by percentiles)...")

# prepare data
flow_rates = df_tubos['flow_rate']
half_widths = df_tubos['width'] * 0.5

# calculate X-axis limits based on percentiles
flow_rate_max_limit = np.percentile(flow_rates, 99.5)
half_width_max_limit = np.percentile(half_widths, 99.5)

# ensure the minimum limit is 0 or very close to 0
flow_rate_min_limit = np.percentile(flow_rates, 0.5) if np.percentile(flow_rates, 0.5) > 0 else 0
half_width_min_limit = np.percentile(half_widths, 0.5) if np.percentile(half_widths, 0.5) > 0 else 0

# create figure with two subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), dpi=100)

# flow rates histogram 
bins_flow_rates = np.linspace(flow_rate_min_limit, flow_rate_max_limit, 20) # More bins for more detail
ax1.hist(flow_rates, bins=bins_flow_rates, color='royalblue', alpha=0.75, edgecolor='black')
ax1.set_title(f'Flow rate distribution')
ax1.set_xlabel('Flow rate (m³/s)')
ax1.set_ylabel('Frequency (count)')
#ax1.set_yscale('log')
ax1.grid(True, linestyle=':', alpha=0.6)



# half-widths histogram 
# Define bins within the limited range
bins_half_widths = np.linspace(half_width_min_limit, half_width_max_limit, 20) # More bins for more detail
ax2.hist(half_widths, bins=bins_half_widths, color='forestgreen', alpha=0.75, edgecolor='black')
ax2.set_title(f'Half-width distribution ')
ax2.set_xlabel('Half-width (m)')
ax2.set_ylabel('Frequency (count)')
#ax2.set_yscale('log')
ax2.grid(True, linestyle=':', alpha=0.6)



# show plot
plt.tight_layout()
fig_path = os.path.join(FIGURES_DIR, 'flow_and_hw_distributions.pdf')
plt.savefig(fig_path, format='pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Porosity Calculation in the ROI

print("Calculating porosity (void fraction) in the ROI...")

# create the box defining the ROI
roi_box = box(roi_xmin, roi_ymin, roi_xmax, roi_ymax)
total_roi_area = roi_box.area

# prepare the ROI for fast intersection queries
prep_roi = prep(roi_box)

# Find the area of all solids WITHIN the ROI
# This list will store the *parts* of the polygons that fall within the ROI
clipped_solids_in_roi = []

for poly in final_circular_polygons:
    # check if the polygon (bead) intersects with the ROI
    if prep_roi.intersects(poly):
        # calculate the intersection (the piece that remains inside)
        clipped_part = poly.intersection(roi_box)
        clipped_solids_in_roi.append(clipped_part)

# Union all pieces to handle overlaps
# .area gives us the total solid area, counting overlaps only once
if clipped_solids_in_roi:
    total_solid_area = unary_union(clipped_solids_in_roi).area
else:
    total_solid_area = 0.0
    
# Calculate porosity
if total_roi_area > 1e-20:
    void_area = total_roi_area - total_solid_area
    porosity = void_area / total_roi_area
    
    print(f"Total ROI area:      {total_roi_area:.2e} m²")
    print(f"Solid area in ROI:   {total_solid_area:.2e} m²")
    print(f"Void Area:           {void_area:.2e} m²")
    print(f"\nROI Porosity (φ):    {porosity * 100:.2f} %")
else:
    print("Error: Total ROI area is zero. Cannot calculate porosity.")

In [ ]:
print("Generating Delaunay triangulation visualization ...")

# Create ghost -> real parent map 
# We need 'near_bottom_indices' and 'near_top_indices' from the previous cell
if 'near_bottom_indices' not in locals():
    print("ERROR: Missing ghost bead data. Run the tube calculation cell first.")
else:
    # This list maps the ghost index (relative, starting at 0) to its real parent index
    ghost_parent_index_map = np.hstack((near_bottom_indices, near_top_indices))
    
    # {global_ghost_index: global_real_parent_index}
    ghost_to_parent_dict = {}
    for k, parent_real_idx in enumerate(ghost_parent_index_map):
        ghost_global_idx = n_real_beads + k
        ghost_to_parent_dict[ghost_global_idx] = parent_real_idx


fig, ax = plt.subplots(figsize=(12, 12), dpi=150)

# Draw the beads (circles)
for poly in final_circular_polygons:
    # Optimization: do not draw polygons that are completely outside the ROI
    if poly.bounds[2] < roi_xmin or poly.bounds[0] > roi_xmax:
        continue
    x, y = poly.exterior.xy
    ax.fill(x, y, alpha=0.3, fc='gray', ec='none', label='Beads' if 'Beads' not in ax.get_legend_handles_labels()[1] else "")

# Draw the centers (real and ghost)
# Filter centers that fall outside the ROI in X
real_centers_in_roi = centers[(centers[:, 0] >= roi_xmin) & (centers[:, 0] <= roi_xmax)]
ax.scatter(real_centers_in_roi[:, 0], real_centers_in_roi[:, 1], c='blue', s=5, label='Real centers', zorder=5)

ghost_centers = all_centers[n_real_beads:]
ghost_centers_in_roi = ghost_centers[(ghost_centers[:, 0] >= roi_xmin) & (ghost_centers[:, 0] <= roi_xmax)]
ax.scatter(ghost_centers_in_roi[:, 0], ghost_centers_in_roi[:, 1], 
           c='black', marker='x', s=15, label='Ghost centers', zorder=5)

# Draw the triangulation edges (WITH FILTER)
for idx1, idx2 in edges:
    p1 = all_centers[idx1]
    p2 = all_centers[idx2]
    
    # Optimization: do not draw edges that are completely outside the ROI
    if (p1[0] < roi_xmin and p2[0] < roi_xmin) or \
       (p1[0] > roi_xmax and p2[0] > roi_xmax):
        continue

    is_idx1_real = (idx1 < n_real_beads)
    is_idx2_real = (idx2 < n_real_beads)
    
    if is_idx1_real and is_idx2_real:
        # Real-real edge
        ax.plot([p1[0], p2[0]], [p1[1], p2[1]], 
                color='gray', linestyle='--', linewidth=0.7, alpha=0.5,
                label='Real-real edge' if 'Real-real edge' not in ax.get_legend_handles_labels()[1] else "")
                
    elif is_idx1_real and not is_idx2_real:
        # Real-ghost edge
        real_idx, ghost_idx = idx1, idx2
        if ghost_idx in ghost_to_parent_dict and real_idx == ghost_to_parent_dict[ghost_idx]:
            ax.plot([p1[0], p2[0]], [p1[1], p2[1]], 
                    color='red', linestyle='-', linewidth=1.0, alpha=0.8,
                    label='Image edge (real-ghost)' if 'Image edge (real-ghost)' not in ax.get_legend_handles_labels()[1] else "")
            
    elif not is_idx1_real and is_idx2_real:
        # Ghost-real edge
        real_idx, ghost_idx = idx2, idx1
        if ghost_idx in ghost_to_parent_dict and real_idx == ghost_to_parent_dict[ghost_idx]:
            ax.plot([p1[0], p2[0]], [p1[1], p2[1]], 
                    color='red', linestyle='-', linewidth=1.0, alpha=0.8,
                    label='Image edge (real-ghost)' if 'Image edge (real-ghost)' not in ax.get_legend_handles_labels()[1] else "")

# Plot configuration 
ax.set_aspect('equal')
ax.set_xlim(roi_xmin, roi_xmax) 
# Y LIMIT extended for ghosts 
ax.set_ylim(ymin - reflection_dist * 0.5, ymax + reflection_dist * 0.5) 

ax.set_title(f"Delaunay triangulation")
ax.set_xlabel("X (m)")
ax.set_ylabel("Y (m)")
ax.legend()
plt.grid(True, linestyle=':', alpha=0.6)
fig_path = os.path.join(FIGURES_DIR, 'delaunay_triangulation.pdf')
plt.savefig(fig_path, format='pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
from collections import Counter

print("Building junctions list...")

# Helper function
def is_in_roi(point):
    return roi_xmin <= point[0] <= roi_xmax

# Create lookup maps from the COMPLETE tube list
edge_to_tube_map_full = {} # Key: (idx1, idx2), Value: tube_id
wall_tube_map = {}         # Key: real_bead_idx, Value: wall_tube_id

print("Creating tube lookup maps (complete)...")
for tube_id, tube_data in df_tubos.iterrows():
    idx1, idx2 = tube_data['bead_indices']
    edge_key = tuple(sorted((idx1, idx2)))
    
    # Map of all edges
    edge_to_tube_map_full[edge_key] = tube_id
    
    # Specific map for wall tubes
    if tube_data['is_wall_tube']:
        real_idx = idx1 if idx1 < n_real_beads else idx2
        wall_tube_map[real_idx] = tube_id

# Set of beads touching a wall ("border" beads)
real_border_bead_indices = set(near_bottom_indices) | set(near_top_indices)

# This will be our new junctions list
# It will contain dictionaries for better structure
junctions_list_structured = []

# Calculate "BULK" junctions
print("Processing 'Bulk' junctions (interior)...")

for simplex in tri.simplices:
    idx1, idx2, idx3 = simplex
    
    # condition: triangle formed ONLY by real beads
    if not (idx1 < n_real_beads and idx2 < n_real_beads and idx3 < n_real_beads):
        continue
        
    c1, c2, c3 = centers[idx1], centers[idx2], centers[idx3]
    roi_count = sum(is_in_roi(c) for c in [c1, c2, c3])
    
    if roi_count == 0:
        continue # discard (no bead in ROI)
    
    # if roi_count > 0, we save the junction
    j_coord = (c1 + c2 + c3) / 3.0
    
    # finding the 3 tubes that form it
    edge1 = tuple(sorted((idx1, idx2)))
    edge2 = tuple(sorted((idx2, idx3)))
    edge3 = tuple(sorted((idx3, idx1)))
    
    tube_ids = [
        edge_to_tube_map_full.get(edge1),
        edge_to_tube_map_full.get(edge2),
        edge_to_tube_map_full.get(edge3)
    ]
    
    # saving only the tubes that exist (width > 0)
    valid_tube_ids = [t_id for t_id in tube_ids if t_id is not None]
    
    if valid_tube_ids: # only saving if it has at least one valid tube
        junctions_list_structured.append({
            'coord': j_coord,
            'tubes': valid_tube_ids,
            'type': 'bulk'
        })

print(f"found {len(junctions_list_structured)} 'bulk' junctions.")

# calculate "wall" junctions
print("processing 'wall' junctions...")
wall_junction_count = 0

for idx1, idx2 in edges: # iterating over all delaunay edges
    
    # condition: both must be "border" beads
    is_border1 = idx1 in real_border_bead_indices
    is_border2 = idx2 in real_border_bead_indices
    
    if not (is_border1 and is_border2):
        continue
        
    # condition: both centers must be WITHIN the roi
    c1, c2 = centers[idx1], centers[idx2]
    if not (is_in_roi(c1) and is_in_roi(c2)):
        continue
        
    # if all conditions are met, find the 3 tubes
    
    # tube 1: the one connecting the two real beads
    tube_bulk_id = edge_to_tube_map_full.get(tuple(sorted((idx1, idx2))))
    
    # tube 2: the wall tube of bead 1
    tube_wall1_id = wall_tube_map.get(idx1)
    
    # tube 3: the wall tube of bead 2
    tube_wall2_id = wall_tube_map.get(idx2)
    
    # making sure all 3 tubes exist (width > 0)
    if tube_bulk_id is None or tube_wall1_id is None or tube_wall2_id is None:
        continue
        
    # calculating junction coordinates
    midpoint1 = df_tubos.loc[tube_wall1_id]['midpoint']
    midpoint2 = df_tubos.loc[tube_wall2_id]['midpoint']
    j_coord = (midpoint1 + midpoint2) / 2.0
    
    # saving the junction
    junctions_list_structured.append({
        'coord': j_coord,
        'tubes': [tube_bulk_id, tube_wall1_id, tube_wall2_id],
        'type': 'wall'
    })
    wall_junction_count += 1

print(f"found {wall_junction_count} 'wall' junctions.")



# to maintain compatibility with your previous code,
# we create the 'junctions_list' only with the tube lists.
junctions_list = [j['tubes'] for j in junctions_list_structured]

print(f"\n--- process completed ---")
print(f"{len(junctions_list)} total junctions found (bulk + wall).")

# analyzing "coordination" (how many tubes per junction)
junction_sizes = [len(j) for j in junctions_list]
size_counts = Counter(junction_sizes)
type_counts = Counter(j['type'] for j in junctions_list_structured)

print("\ndistribution by type:")
for j_type, count in type_counts.items():
    print(f"  - type '{j_type}' junctions: {count}")

print("\ntube distribution per junction:")
for size, count in sorted(size_counts.items()):
    print(f"  - junctions with {size} tubes: {count}")

# showing the first 5 junctions
print("\nexample (first 5 junctions):")
for i, j_data in enumerate(junctions_list_structured[:5]):
    print(f"  junction {i} (type {j_data['type']}): tubes {j_data['tubes']}")

In [ ]:
print("filtering the tube list and junctions by roi...")

# filter the main tube list (df_tubos)
# (the current df_tubos is the 'full' one)
midpoints = np.stack(df_tubos['midpoint'].values)
in_roi_mask = (midpoints[:, 0] >= roi_xmin) & (midpoints[:, 0] <= roi_xmax)

# this is the new filtered dataframe you wanted
df_tubos_roi = df_tubos[in_roi_mask].copy()

print(f"filtered tube list: {len(df_tubos)} -> {len(df_tubos_roi)} tubes in roi.")

# get ids (indices) of tubes that ARE in the roi
valid_tube_ids = set(df_tubos_roi.index)

# filter the junctions list (junctions_list_structured)
junctions_list_filtered = []
junctions_list_structured_filtered = []

for junction in junctions_list_structured:
    # filter the tube list of this junction
    filtered_tubes = [
        tube_id for tube_id in junction['tubes'] 
        if tube_id in valid_tube_ids
    ]
    
    # do not save junctions with < 2 tubes
    # (a real junction must connect at least 2 tubes)
    if len(filtered_tubes) > 1:
        new_junction_data = junction.copy()
        new_junction_data['tubes'] = filtered_tubes
        
        junctions_list_structured_filtered.append(new_junction_data)
        junctions_list_filtered.append(filtered_tubes)

print(f"filtered junctions list: {len(junctions_list_structured)} -> {len(junctions_list_structured_filtered)} valid junctions.")

# replace old variables with new filtered ones
# from now on, df_tubos and junctions_list will be filtered by roi.
df_tubos = df_tubos_roi
junctions_list = junctions_list_filtered
junctions_list_structured = junctions_list_structured_filtered

print("'df_tubos' and 'junctions_list' variables updated.")

In [ ]:
from collections import Counter

print("refining network topology (handling 'loose ends')...")

# we assume 'df_tubos' and 'junctions_list_structured' are already filtered by the roi.

# count occurrences of each tube in the junctions
# we count how many junctions "touch" each tube
tube_appearance_count = Counter()
for junction in junctions_list_structured:
    for tube_id in junction['tubes']:
        tube_appearance_count[tube_id] += 1

# identify "loose ends" and classify them 
# (tubos that appear 0 or 1 time)

new_junctions_to_add = []
tubes_to_delete = set()

# define a tolerance for the boundaries (2% of the roi width)
roi_width = roi_xmax - roi_xmin
tolerance = roi_width * 0.02
left_boundary = roi_xmin + tolerance
right_boundary = roi_xmax - tolerance

print(f"boundary tolerance in x defined as: {tolerance:.4f} m")

# iterate over ALL tubes in our roi-filtered list
for tube_id in df_tubos.index:
    count = tube_appearance_count.get(tube_id, 0)
    
    # look for tubes with 0 or 1 connection
    if count == 0 or count == 1:
        
        # it is a "loose end". we create a candidate junction.
        midpoint = df_tubos.loc[tube_id]['midpoint']
        j_coord = midpoint
        
        # check if it is an inlet/outlet (near the boundary)
        is_near_left = (j_coord[0] <= left_boundary)
        is_near_right = (j_coord[0] >= right_boundary)
        
        if is_near_left or is_near_right:
            # it is a VALID inlet/outlet.
            # we create the new 1-tube junction.
            new_junctions_to_add.append({
                'coord': j_coord,
                'tubes': [tube_id],
                'type': 'outlet' 
            })
        else:
            # it is an INVALID "dangling tube" (in the middle of the bulk).
            # we mark it for deletion.
            tubes_to_delete.add(tube_id)

print(f"identified {len(new_junctions_to_add)} inlet/outlet tubes.")
print(f"identified {len(tubes_to_delete)} internal 'dangling' tubes to delete.")

# filter the lists 

# remove 'dangling' tubes from the main list
df_tubos_final = df_tubos.drop(index=list(tubes_to_delete))

# filter the existing junctions list
final_junctions_structured = []
for junction in junctions_list_structured:
    
    # recreate the junction's tube list,
    # omitting those marked for deletion
    new_tube_list = [
        t_id for t_id in junction['tubes'] 
        if t_id not in tubes_to_delete
    ]
    
    # we only keep the junction if it still has tubes
    if len(new_tube_list) > 0:
        junction['tubes'] = new_tube_list
        final_junctions_structured.append(junction)

# add the new inlet/outlet junctions
final_junctions_structured.extend(new_junctions_to_add)

# replace global variables 

df_tubos = df_tubos_final
junctions_list_structured = final_junctions_structured
junctions_list = [j['tubes'] for j in junctions_list_structured] # regenerate the simple list

print(f"\n--- cleaning completed ---")
print(f"final tube list: {len(df_tubos)}")
print(f"final junction list: {len(junctions_list)}")

# see the new type distribution
final_type_counts = Counter(j['type'] for j in junctions_list_structured)
print("\nnew junction type distribution:")
for j_type, count in final_type_counts.items():
    print(f"  - type '{j_type}' junctions: {count}")

In [ ]:
from collections import Counter
import numpy as np

print("starting final 'border-to-border' filter...")

# identifying border junctions 

# redefining tolerance (same as in the previous cell)
roi_width = roi_xmax - roi_xmin
tolerance = roi_width * 0.02
left_boundary = roi_xmin + tolerance
right_boundary = roi_xmax - tolerance

# saving the indices of the 'junctions_list_structured' list
border_junction_indices = set()
print(f"marking 'border' junctions (x tolerance: {tolerance:.4f} m)...")

for j_index, junction in enumerate(junctions_list_structured):
    coord = junction['coord']
    if coord[0] <= left_boundary or coord[0] >= right_boundary:
        border_junction_indices.add(j_index)

print(f"{len(border_junction_indices)} 'border' junctions identified.")

# creating inverse map (tube -> junctions) ---
# we need to know which two junctions each tube connects

# (using current 'junctions_list_structured' and 'df_tubos')
tube_to_junctions_map = {tube_id: [] for tube_id in df_tubos.index}

for j_index, junction in enumerate(junctions_list_structured):
    for tube_id in junction['tubes']:
        # making sure the tube exists in the map
        if tube_id in tube_to_junctions_map:
            tube_to_junctions_map[tube_id].append(j_index)

# identifying "border-to-border" tubes 
tubes_to_delete_b2b = set()

for tube_id, connected_junction_indices in tube_to_junctions_map.items():
    
    # we are only interested in tubes connecting EXACTLY 2 junctions
    if len(connected_junction_indices) == 2:
        j_idx_1 = connected_junction_indices[0]
        j_idx_2 = connected_junction_indices[1]
        
        # if BOTH junctions are "border" junctions, we mark the tube for deletion
        if j_idx_1 in border_junction_indices and j_idx_2 in border_junction_indices:
            tubes_to_delete_b2b.add(tube_id)

print(f"{len(tubes_to_delete_b2b)} 'border-to-border' tubes identified for removal.")

# applying filters and cleaning

# removing tubes from the main dataframe
df_tubos_final_b2b = df_tubos.drop(index=list(tubes_to_delete_b2b))

# filtering the junctions list (removing tubes and empty junctions)
final_junctions_structured_b2b = []
for j_index, junction in enumerate(junctions_list_structured):
    
    # recreating the junction's tube list,
    # omitting those we just deleted
    new_tube_list = [
        t_id for t_id in junction['tubes'] 
        if t_id not in tubes_to_delete_b2b
    ]
    
    # if the junction has no tubes left, we remove it
    if len(new_tube_list) > 0:
        junction['tubes'] = new_tube_list
        final_junctions_structured_b2b.append(junction)

# replacing global variables
df_tubos = df_tubos_final_b2b
junctions_list_structured = final_junctions_structured_b2b
junctions_list = [j['tubes'] for j in junctions_list_structured] # regenerating simple list

# recalculating border_junction_indices
# (because the list size has changed and indices have shifted)
print("recalculating final 'border_junction_indices' set...")
border_junction_indices = set()
for j_index, junction in enumerate(junctions_list_structured):
    coord = junction['coord']
    if coord[0] <= left_boundary or coord[0] >= right_boundary:
        border_junction_indices.add(j_index)

print(f"\n--- 'border-to-border' filter completed ---")
print(f"final tube list: {len(df_tubos)}")
print(f"final junction list: {len(junctions_list)}")
print(f"border junction set updated to {len(border_junction_indices)}.")

In [ ]:
import matplotlib.pyplot as plt

print("Generating pore network visualization (junctions and tubes)...")

# we assume 'final_circular_polygons', 'df_tubos', 
# 'junctions_list_structured', 'roi_xmin', 'roi_xmax', 
# 'ymin', 'ymax' are defined and filtered.

# create a lookup map for tube midpoints
# (using the df_tubos already filtered by roi)
tube_midpoints_map = {idx: row['midpoint'] for idx, row in df_tubos.iterrows()}

# create the figure with high resolution
fig, ax = plt.subplots(figsize=(14, 14), dpi=200) 

# drawing the beads (circles) in the background
for poly in final_circular_polygons:
    # optimization: do not draw polygons completely outside the roi
    if poly.bounds[2] < roi_xmin or poly.bounds[0] > roi_xmax:
        continue
    x, y = poly.exterior.xy
    ax.fill(x, y, alpha=0.3, fc='gray', ec='none', label='Beads' if 'Beads' not in ax.get_legend_handles_labels()[1] else "")

# prepare lists to draw junctions and connections
junction_coords = []
connection_lines = [] # list of [[x_j, y_j], [x_t, y_t]]

# we iterate over the structured list, which contains the coordinates
for junction in junctions_list_structured:
    j_center = junction['coord']
    tube_ids = junction['tubes']
    
    # saving the junction center
    junction_coords.append(j_center)
    
    # saving the connection lines
    for tube_id in tube_ids:
        # looking for the tube midpoint (should exist as everything is filtered)
        midpoint = tube_midpoints_map.get(tube_id)
        if midpoint is not None:
            connection_lines.append([j_center, midpoint])

# drawing the connections (green lines)
for line in connection_lines:
    p_junction = line[0]
    p_midpoint = line[1]
    ax.plot([p_junction[0], p_midpoint[0]], [p_junction[1], p_midpoint[1]], 
            color='lime', linestyle='-', linewidth=1.0, alpha=0.8, zorder=8,
            label='Tube' if 'Tube' not in ax.get_legend_handles_labels()[1] else "")

# drawing the junction centers (purple dots)
if junction_coords:
    junction_coords_np = np.array(junction_coords)
    ax.scatter(junction_coords_np[:, 0], junction_coords_np[:, 1], 
               color='purple', 
               s=10, 
               marker='o', 
               zorder=10, 
               label='Junction' if 'Junction' not in ax.get_legend_handles_labels()[1] else "")


# plot configuration
ax.set_aspect('equal')
ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax) 
ax.set_title(f"Pore network visualization")
ax.set_xlabel("X (m)")
ax.set_ylabel("Y (m)")
legend = ax.legend(
    loc='best', 
    framealpha=1.0,    
    facecolor='white', 
    edgecolor='black'
)
legend.set_zorder(100)

# Importante para que la ventana se ajuste y no recorte la leyenda
plt.tight_layout()
plt.grid(True, linestyle=':', alpha=0.6)
fig_path = os.path.join(FIGURES_DIR, 'pore_network_picture.pdf')
plt.savefig(fig_path, format='pdf', dpi=300, bbox_inches='tight')
plt.show()

print(f"Visualization completed. {len(junction_coords)} junctions in the ROI.")

In [ ]:
import numpy as np
tube_flow_map = {idx: row['flow_rate'] for idx, row in df_tubos.iterrows()}

# Map of tubes -> [junction_idx_A, junction_idx_B]
tube_to_junctions_map = {tube_id: [] for tube_id in df_tubos.index}
for j_index, junction_tubes in enumerate(junctions_list):
    for tube_id in junction_tubes:
        if tube_id in tube_to_junctions_map:
            tube_to_junctions_map[tube_id].append(j_index)

# verify  N_SAMPLES exists
if 'N_SAMPLES' not in locals():
    N_SAMPLES = 50

# compute direction of each tube
tube_direction_map = {}

for tube_id, tube_data in df_tubos.iterrows():
    
    # find the two junctions of each tube
    junction_indices = tube_to_junctions_map.get(tube_id, [])
    if len(junction_indices) != 2:
        continue
        
    j_idx_A, j_idx_B = junction_indices
    j_coord_A = junctions_list_structured[j_idx_A]['coord']
    j_coord_B = junctions_list_structured[j_idx_B]['coord']

    # recompute geometry
    idx1, idx2 = tube_data['bead_indices']
    is_wall_tube = tube_data['is_wall_tube']
    width = tube_data['width']
    
    if is_wall_tube:
        real_idx = idx1 if idx1 < n_real_beads else idx2
        c_real = centers[real_idx]
        r_real = radii[real_idx]
        is_top_wall = c_real[1] > (ymax + ymin) / 2.0
        
        if is_top_wall:
            p_edge_bead = np.array([c_real[0], c_real[1] + r_real])
            p_edge_wall = np.array([c_real[0], ymax])
            n_vec_path = np.array([0.0, 1.0])
        else:
            p_edge_bead = np.array([c_real[0], c_real[1] - r_real])
            p_edge_wall = np.array([c_real[0], ymin])
            n_vec_path = np.array([0.0, -1.0])
    else:
        c1, c2 = all_centers[idx1], all_centers[idx2]
        r1, r2 = all_radii[idx1], all_radii[idx2]
        center_vec = c2 - c1
        dist = np.linalg.norm(center_vec)
        n_vec_path = center_vec / dist
        p_edge_wall = c1 + n_vec_path * r1
        p_edge_bead = c2 - n_vec_path * r2
        
    # compute oriented tangent
    n_vec_flow_initial = np.array([-n_vec_path[1], n_vec_path[0]])
    t_oriented = n_vec_flow_initial

    if tree is not None and width > 1e-9:
        sample_points_2d = np.linspace(p_edge_wall, p_edge_bead, N_SAMPLES)
        dists, idxs = tree.query(sample_points_2d, k=1)
        velocities_2d = U_cells[idxs]
        velocities_2d = np.nan_to_num(velocities_2d, nan=0.0, posinf=0.0, neginf=0.0)
        
        proj_mean = np.dot(velocities_2d, n_vec_flow_initial).mean()
        sign = np.sign(proj_mean) if np.sign(proj_mean) != 0 else 1.0
        t_oriented = n_vec_flow_initial * sign
    
    # determine intial and final junction
    midpoint = tube_data['midpoint'] 
    vec_m_to_A = j_coord_A - midpoint
    dp_A = t_oriented.dot(vec_m_to_A)
    
    if dp_A > 0:
        j_initial_id, j_final_id = j_idx_B, j_idx_A
    else:
        j_initial_id, j_final_id = j_idx_A, j_idx_B
        
    tube_direction_map[tube_id] = (j_initial_id, j_final_id)

print(f"Direction computed for {len(tube_direction_map)} tubes.")

# compute flow in and flow out for each junction
print("Computing in/out flows for each junction...")

junction_flow_in = {j: 0.0 for j in range(len(junctions_list_structured))}
junction_flow_out = {j: 0.0 for j in range(len(junctions_list_structured))}

for tube_id, (j_initial, j_final) in tube_direction_map.items():
    flow = tube_flow_map.get(tube_id, 0.0)
    junction_flow_out[j_initial] += flow
    junction_flow_in[j_final] += flow



In [ ]:
import numpy as np
import matplotlib.pyplot as plt

print("Generating 'flow in' vs 'flow out' plots for each junction...")

# Using pre-calculated 'junction_flow_in' and 'junction_flow_out' maps 

all_flow_in = []
all_flow_out = []

# iterating over all junctions
for junction_id in range(len(junctions_list_structured)):
    
    # ignore border junctions (inlets/outlets)
    if junction_id in border_junction_indices:
        continue
        
    all_flow_in.append(junction_flow_in[junction_id])
    all_flow_out.append(junction_flow_out[junction_id])

# converting to numpy arrays
flow_in_np = np.array(all_flow_in)
flow_out_np = np.array(all_flow_out)

print(f"Calculation completed for {len(flow_in_np)} internal junctions.")

# generating plots

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 7), dpi=100)

# plot 1: scatter plot (flow out vs flow in) 

# calculating a visual limit
if len(flow_in_np) > 0:
    combined_max = np.percentile(np.hstack((flow_in_np, flow_out_np)), 99.5)
    if combined_max == 0: 
        combined_max = np.max(np.hstack((flow_in_np, flow_out_np)))
else:
    combined_max = 1.0 # default value

ax1.scatter(flow_out_np, flow_in_np, alpha=0.4, s=10, label='internal junctions')
ax1.plot([0, combined_max], [0, combined_max], 'r--', linewidth=2, label='y=x line')
ax1.set_xlabel('Flow out (m³/s)') 
ax1.set_ylabel('Flow in (m³/s)')  
ax1.set_title('Masse conservation (internal junctions)')
ax1.set_aspect('equal')
ax1.set_xlim(0, combined_max)
ax1.set_ylim(0, combined_max)
ax1.legend()
ax1.grid(True, linestyle=':', alpha=0.6)

# plot 2: normalized pdfs 

bins = np.linspace(0, combined_max, 50)

ax2.hist(flow_out_np, bins=bins, density=True, alpha=0.7, color='blue', label='PDF flow out') 
ax2.hist(flow_in_np, bins=bins, density=True, alpha=0.7, color='red', label='PDF flow in')   
ax2.set_xlabel('Flow rate (m³/s)')
ax2.set_ylabel('Probability density (normalized)')
ax2.set_title('Flow distribution (internal junctions)')
ax2.legend()
ax2.grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()
fig_path = os.path.join(FIGURES_DIR, 'mass_conservation_figure.pdf')
plt.savefig(fig_path, format='pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

print("Identifying and visualizing outlier junctions...")

    
RELATIVE_TOLERANCE = 0.96 
ABSOLUTE_TOLERANCE = 1e-20 


outlier_junctions_data = []


for junction_id, junction in enumerate(junctions_list_structured):
    
    if junction_id in border_junction_indices:
        continue

    tube_id_list = junction['tubes']
    flows = [tube_flow_map[t_id] for t_id in tube_id_list if t_id in tube_flow_map]
    
    if len(flows) <= 1:
        continue
    
    flow_big = max(flows)
    flow_smalls = sum(flows) - flow_big
    
    # if it's an outlier, we save the whole junction (which has 'coord' and 'tubes')
    if flow_big > ABSOLUTE_TOLERANCE and flow_smalls < (flow_big * RELATIVE_TOLERANCE):
        outlier_junctions_data.append(junction)

print(f"{len(outlier_junctions_data)} outlier junctions found for visualization.")

# create visualization 
# (we assume 'final_circular_polygons', 'roi_xmin', 'roi_xmax', 'ymin', 'ymax' exist)

fig, ax = plt.subplots(figsize=(14, 14), dpi=150)

# drawing the beads (background)
for poly in final_circular_polygons:
    # drawing optimization
    if poly.bounds[2] < roi_xmin or poly.bounds[0] > roi_xmax:
        continue
    x, y = poly.exterior.xy
    ax.fill(x, y, alpha=0.3, fc='gray', ec='none', label='Beads' if 'Beads' not in ax.get_legend_handles_labels()[1] else "")

# prepare lists to draw outlier junctions and connections
outlier_centers_coords = []
outlier_connection_lines = [] # list of [[x_j, y_j], [x_t, y_t]]

for junction in outlier_junctions_data:
    j_center = junction['coord']
    tube_ids = junction['tubes']
    
    outlier_centers_coords.append(j_center)
    
    for tube_id in tube_ids:
        midpoint = tube_midpoints_map.get(tube_id) # using the map from the previous cell
        if midpoint is not None:
            outlier_connection_lines.append([j_center, midpoint])

# drawing connections (red lines)
for line in outlier_connection_lines:
    p_junction = line[0]
    p_midpoint = line[1]
    ax.plot([p_junction[0], p_midpoint[0]], [p_junction[1], p_midpoint[1]], 
            color='red', linestyle='-', linewidth=2.0, alpha=1.0, zorder=10,
            label='Outlier connection' if 'Outlier connection' not in ax.get_legend_handles_labels()[1] else "")

# drawing junction centers (red circles)
if outlier_centers_coords:
    coords_np = np.array(outlier_centers_coords)
    ax.scatter(coords_np[:, 0], coords_np[:, 1], 
               color='red', 
               s=30,  
               marker='o', 
               edgecolor='black', 
               zorder=12, 
               label='Outlier junction' if 'Outlier junction' not in ax.get_legend_handles_labels()[1] else "")

# plot configuration
ax.set_aspect('equal')
ax.set_xlim(roi_xmin, roi_xmax)
ax.set_ylim(ymin, ymax) 
ax.set_title(f"Flow imbalance junction visualization ({len(outlier_junctions_data)} outliers)")
ax.set_xlabel("x (m)")
ax.set_ylabel("y (m)")
ax.axvline(x=left_boundary, color='cyan', linestyle=':', linewidth=2, label='Boundary limit')
ax.axvline(x=right_boundary, color='cyan', linestyle=':', linewidth=2)


legend = ax.legend(
    loc='best',
    framealpha=1.0,
    facecolor='white',
    edgecolor='black'
)
legend.set_zorder(100)

plt.grid(True, linestyle=':', alpha=0.6)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree

print("Starting pressure gradient computation...")

# Prepare pressure sampling (similar to velocity) ---
RHO_FLUID = 1000.0 # Water density (kg/m^3)
P_NAME_CELL = None

# Search for pressure field (OpenFOAM uses 'p' or 'p_rgh')
common_names = ['p', 'pressure', 'Pressure', 'p_rgh']
for name in common_names:
    if name in vol_mesh.cell_data:
        P_NAME_CELL = name
        break

if P_NAME_CELL is None:
    print("FATAL ERROR: pressure field ('p' or 'p_rgh') not found in vol_mesh.cell_data.")
    # If not found, we stop this cell
    pressure_gradients_np = np.array([0]) # Empty array to prevent plot failure
else:
    print(f"Using cell pressure field: '{P_NAME_CELL}'")
    
    # Extract data and build the tree
    # (We use the same cell centers as for velocity)
    if 'cell_centers_3d' not in locals():
         cell_centers_3d = vol_mesh.cell_centers().points
         
    p_data_raw = vol_mesh.cell_data[P_NAME_CELL] # (M,)
    
    # Build the 2D tree
    tree_p = cKDTree(cell_centers_3d[:, :2])
    print("KDTree for pressure built.")

    # Compute pressure at EACH junction 
    
    # Get coordinates of all junctions
    junction_coords = np.array([j['coord'] for j in junctions_list_structured])
    
    # Query the tree to find the closest cell to each junction
    dists, idxs = tree_p.query(junction_coords, k=1)
    
    # Get pressure values (p/rho) and correct them
    p_junction_raw = p_data_raw[idxs]
    p_junction_corrected = p_junction_raw * RHO_FLUID # Real pressure in Pascals
    
    # Create a map for easy lookup: {junction_id: Pressure}
    junction_pressure_map = {j_idx: p for j_idx, p in enumerate(p_junction_corrected)}
    print(f"Pressure sampled and corrected for {len(junction_pressure_map)} junctions.")

    # Calculate the gradient for EACH tube
    pressure_gradients = []
    
    # We iterate over the direction map (contains j_initial and j_final)
    for tube_id, (j_initial_id, j_final_id) in tube_direction_map.items():
        
        # Get pressures of connected junctions
        p_initial = junction_pressure_map.get(j_initial_id)
        p_final = junction_pressure_map.get(j_final_id)
        
        if p_initial is None or p_final is None:
            continue
            
        # Calculate tube length (L) (distance between junctions)
        coord_initial = junctions_list_structured[j_initial_id]['coord']
        coord_final = junctions_list_structured[j_final_id]['coord']
        length = np.linalg.norm(coord_final - coord_initial)
        
        if length < 1e-20: # Avoid division by zero
            continue
            
        # Calculate pressure gradient
        # (We use the absolute value, since the sign depends on the direction)
        delta_p = p_final - p_initial
        gradient = np.abs(delta_p) / length # |(P2 - P1) / L|
        
        pressure_gradients.append(gradient)

    # Convert to NumPy array for statistics
    pressure_gradients_np = np.array(pressure_gradients)


# Calculate statistics 
if pressure_gradients_np.size > 0:
    mean_grad = np.mean(pressure_gradients_np)
    var_grad = np.var(pressure_gradients_np)
    std_dev_grad = np.std(pressure_gradients_np)

    print("\n--- Pressure gradient statistics ---")
    print(f"Mean:\t\t {mean_grad:.2f} Pa/m")
    print(f"Variance:\t {var_grad:.2f} (Pa/m)^2")
    print(f"Std. deviation:\t {std_dev_grad:.2f} Pa/m")
else:
    print("No pressure gradients were calculated.")
    mean_grad = 0

# Plot the histogram (normalized)
print("Generating histogram (PDF)...")

fig, ax = plt.subplots(figsize=(10, 6), dpi=100)

if pressure_gradients_np.size > 0:
    # Limit histogram range to exclude extreme outliers
    p_min = np.percentile(pressure_gradients_np, 0.5)
    p_max = np.percentile(pressure_gradients_np, 99.5)
    
    # Ensure the range is valid
    if p_max <= p_min:
        p_min = np.min(pressure_gradients_np)
        p_max = np.max(pressure_gradients_np)
    
    bins = np.linspace(p_min, p_max, 50)
    

    ax.hist(pressure_gradients_np, bins=bins, color='darkorange', alpha=0.75, edgecolor='black', density=True)
    
    # Mean line
    ax.axvline(mean_grad, color='red', linestyle='--', linewidth=2, 
                label=f'Mean: {mean_grad:.2f} Pa/m')
    
    ax.set_title('Pressure gradient distribution (PDF)', fontsize=16) 
    ax.set_xlabel('Pressure gradient (Pa/m)', fontsize=12)
    ax.set_ylabel('Probability density', fontsize=12) 
    ax.legend()
    ax.grid(True, linestyle=':', alpha=0.6)
    ax.ticklabel_format(axis='x', style='sci', scilimits=(0,0))
else:
    ax.set_title('Pressure gradient distribution (no data)', fontsize=16)
    ax.set_xlabel('Pressure gradient (Pa/m)', fontsize=12)
    ax.set_ylabel('Probability density', fontsize=12) 

plt.tight_layout()
fig_path = os.path.join(FIGURES_DIR, 'pressure_gradient_distribution.pdf')
plt.savefig(fig_path, format='pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import numpy as np

print("starting the network export to files...")

# Save "junction_coordinates.txt" (with node ID and pressure) 
print("saving junction_coordinates.txt...")

# retrieve coordinates
junction_coords = np.array([j['coord'] for j in junctions_list_structured])
node_ids = np.arange(len(junction_coords))
node_ids_col = node_ids.reshape(-1, 1)

# retrieve pressures (already converted to pascals in the "pressure gradient" cell)
if 'junction_pressure_map' in locals():
    # making sure the order matches the junction index
    pressures_list = [junction_pressure_map.get(i, 0.0) for i in range(len(junctions_list_structured))]
else:
    print("WARNING: pressure map not found. zeros will be saved.")
    pressures_list = [0.0] * len(junctions_list_structured)

pressures_col = np.array(pressures_list).reshape(-1, 1)

# stack: [ID, X, Y, Pressure]
output_array = np.hstack((node_ids_col, junction_coords, pressures_col))
coord_path = os.path.join(RAW_DATA_DIR, "junction_coordinates.txt")
np.savetxt(
    coord_path, 
    output_array, 
    fmt=['%d', '%.8f', '%.8f', '%.6e'], # ID(int), X(float), Y(float), P(scientific)
    header="NodeID X_m Y_m Pressure_Pa",
    comments=""
)
print(f"{len(output_array)} nodes with pressure saved.")


# preparations for "junction_links.txt"

print("Pre-calculating outputs for stagnation nodes...")
stagnation_node_out_counts = {}

# Identify stagnation nodes (flow_out <= 1e-20)
stagnation_node_ids = set()
if 'junction_flow_out' in locals():
    for j_index in range(len(junctions_list_structured)):
        if junction_flow_out.get(j_index, 0.0) <= 1e-20:
            stagnation_node_ids.add(j_index)

# Count topological outputs
node_out_link_count = {j_id: 0 for j_id in range(len(junctions_list_structured))}
for tube_id, (j_init, j_fin) in tube_direction_map.items():
    if j_init in node_out_link_count:
        node_out_link_count[j_init] += 1

for j_id in stagnation_node_ids:
    stagnation_node_out_counts[j_id] = node_out_link_count.get(j_id, 0)

# Generate links
print("generating links with physical properties...")
links_data = []

for tube_id, (j_initial_id, j_final_id) in tube_direction_map.items():
    
    # flow and weight data
    flow_this_tube = tube_flow_map.get(tube_id, 0.0)
    flow_out_initial = junction_flow_out.get(j_initial_id, 0.0)
    
    # get the width from the original dataframe using the tube_id
    width = df_tubos.loc[tube_id]['width']
    half_width = width / 2.0
    
    weight = 0.0
    
    if flow_out_initial > 1e-20:
        # normal case
        weight = flow_this_tube / flow_out_initial
        weight = min(weight, 1.0)
    else:
        # stagnation case (equiprobable)
        N_out = stagnation_node_out_counts.get(j_initial_id, 0)
        if N_out > 0:
            weight = 1.0 / N_out
        else:
            weight = 0.0

    # save: [In, Out, Weight, HalfWidth, FlowRate]
    links_data.append([j_initial_id, j_final_id, weight, half_width, flow_this_tube])

# save "junction_links.txt" 
print("saving junction_links.txt...")
links_array = np.array(links_data)
link_path = os.path.join(RAW_DATA_DIR, 'junction_links.txt')
np.savetxt(
    link_path,
    links_array,
    # format: Int, Int, Float, Scientific(geo), Scientific(flow)
    fmt=['%d', '%d', '%.8f', '%.8e', '%.8e'], 
    header="NodeIn NodeOut Weight HalfWidth_m FlowRate_m3s",
    comments=""
)

print(f"\n--- export completed ---")
print(f"Generated files: junction_coordinates.txt (with P), junction_links.txt (with a and Q)")

In [ ]:
import numpy as np

print("Comparing analytical vs real average pore flow rate...")

# identify actual topological inlets on the left boundary
left_limit = roi_xmin + tolerance
real_inlets = [
    i for i in border_junction_indices 
    if junctions_list_structured[i]['coord'][0] <= left_limit
]
print(f"real inlets detected: {len(real_inlets)}")

# calculate total flow entering the network
total_q_in = sum(junction_flow_out[idx] for idx in real_inlets)

# gather geometric properties
domain_width = roi_ymax - roi_ymin
mean_r2 = np.mean(radii**2)

# analytical estimation (corrected formula based on disk density, no factor of 2)
sqrt_term = np.sqrt((np.pi * mean_r2) / (1.0 - porosity))
q_p_analytical = (total_q_in / domain_width) * sqrt_term

# calculate real average flow in internal pores (bulk)
internal_pore_flows = [
    junction_flow_in[idx] 
    for idx in range(len(junctions_list_structured))
    if idx not in border_junction_indices
]
q_p_real = np.mean(internal_pore_flows) if internal_pore_flows else 0.0

# display the results cleanly
print("\n--- average pore flow rate (<Qp>) ---")
print(f"total inlet flow:    {total_q_in:.4e} m³/s")
print(f"internal pores:      {len(internal_pore_flows)}")
print("-" * 40)
print(f"analytical estimate: {q_p_analytical:.4e} m³/s")
print(f"real network value:  {q_p_real:.4e} m³/s")

if q_p_real > 0:
    error = abs(q_p_analytical - q_p_real) / q_p_real
    print(f"relative error:      {error * 100:.2f} %")